# 14 — DINOv3 + SAM2 Multi-Source Ensemble (Lea County, NM)

Focused ensemble using only DINOv3 (fine-tuned) across multiple satellite
sources, with per-source SAM2 boundary refinement using each sensor's
native raster.

**Pipeline per source:** Download composite → Fine-tune DINOv3 → Inference → SAM2 refinement
**Then:** Majority-vote ensemble → Evaluate against NMOSE reference

**Estimated runtime:** ~1–2 hours (GPU recommended)

**Prerequisites:**
```bash
pip install agribound[gee,geoai,samgeo]
agribound auth --project YOUR_GEE_PROJECT
```

## Configuration

In [ ]:
import json
import logging
import warnings
from pathlib import Path

import geopandas as gpd

import agribound
from agribound.evaluate import evaluate

warnings.filterwarnings("ignore", message=".*organizePolygons.*")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(message)s",
    datefmt="%H:%M:%S",
)
logging.getLogger("urllib3").setLevel(logging.CRITICAL)
logging.getLogger("googleapiclient").setLevel(logging.CRITICAL)
logging.getLogger("geedim").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.WARNING)

GEE_PROJECT = "YOUR_GEE_PROJECT"  # <-- Replace with your GEE project ID

NMOSE_SHAPEFILE = "../../examples/NMOSE Field Boundaries/WUCB ag polys.shp"
OUTPUT_DIR = Path("../../outputs/lea_county_dinov3_sam2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CRS = "EPSG:26913"

COUNTY_CODE = "25"
FINE_TUNE_EPOCHS = 30
BATCH_SIZE = 8
YEARS = [2022]  # Single year for notebook (script runs 2020-2022)
VOTE_THRESHOLD = 0.3

SAM_REFINE = True
SAM_MODEL = "large"
SAM_BATCH_SIZE = 100

SOURCES = ["sentinel2", "landsat", "hls", "naip", "spot"]

SOURCE_YEAR_RANGE = {
    "sentinel2": (2017, 2025),
    "landsat": (1985, 2025),
    "hls": (2013, 2025),
    "naip": (2003, 2025),
    "spot": (2012, 2023),
}

## Create Study Area

In [ ]:
gdf = gpd.read_file(NMOSE_SHAPEFILE)
county_gdf = gdf[gdf["County"] == COUNTY_CODE].copy()
county_4326 = county_gdf.to_crs(epsg=4326)
bounds = county_4326.total_bounds
bbox_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [bounds[0], bounds[1]],
                        [bounds[2], bounds[1]],
                        [bounds[2], bounds[3]],
                        [bounds[0], bounds[3]],
                        [bounds[0], bounds[1]],
                    ]
                ],
            },
            "properties": {"name": f"Lea County (County {COUNTY_CODE})"},
        }
    ],
}
study_area_path = str(OUTPUT_DIR / "study_area.geojson")
with open(study_area_path, "w") as f:
    json.dump(bbox_geojson, f)

ref_path = OUTPUT_DIR / "reference.gpkg"
if not ref_path.exists():
    county_gdf.to_file(ref_path, driver="GPKG", layer="fields")
ref_path = str(ref_path)

print(f"Study area: Lea County ({len(county_gdf)} reference polygons)")

## Phase 1: DINOv3 + SAM2 Per Source

In [ ]:
from agribound.config import AgriboundConfig
from agribound.engines.samgeo_engine import refine_boundaries
from agribound.postprocess.simplify import simplify_polygons, smooth_polygons

all_results = {}

for year in YEARS:
    print(f"\n--- Year {year} ---")
    all_results[year] = {}

    for source in SOURCES:
        yr_min, yr_max = SOURCE_YEAR_RANGE[source]
        if year < yr_min or year > yr_max:
            continue

        output_path = OUTPUT_DIR / f"fields_dinov3_{source}_{year}.gpkg"
        lulc_path = OUTPUT_DIR / f"fields_dinov3_{source}_{year}_lulc.gpkg"
        pipeline_path = OUTPUT_DIR / f"fields_dinov3_{source}_{year}_pipeline.gpkg"
        tag = f"{source}/dinov3"
        print(f"  {tag}: starting...", flush=True)

        try:
            if output_path.exists():
                source_gdf = gpd.read_file(output_path)
            else:
                kwargs = dict(
                    study_area=study_area_path,
                    source=source,
                    year=year,
                    engine="dinov3",
                    output_path=str(pipeline_path),
                    gee_project=GEE_PROJECT,
                    min_area=2500,
                    simplify=2.0,
                    device="auto",
                    reference_boundaries=ref_path,
                    fine_tune=True,
                    fine_tune_epochs=FINE_TUNE_EPOCHS,
                    engine_params={"batch_size": BATCH_SIZE},
                )
                if source in ("sentinel2", "landsat", "hls"):
                    kwargs["composite_method"] = "median"
                    kwargs["cloud_cover_max"] = 20
                    kwargs["date_range"] = (f"{year}-10-01", f"{year}-10-31")
                elif source == "spot":
                    kwargs["composite_method"] = "median"
                    kwargs["cloud_cover_max"] = 15
                elif source == "naip":
                    kwargs["min_area"] = 5000

                source_gdf = agribound.delineate(**kwargs)

                if source_gdf.crs is not None and str(source_gdf.crs) != OUTPUT_CRS:
                    source_gdf = source_gdf.to_crs(OUTPUT_CRS)

                # Save LULC-filtered, pre-SAM result
                source_gdf.to_file(lulc_path, driver="GPKG", layer="fields")
                print(f"    LULC-filtered: {len(source_gdf)} fields")

                # SAM2 refinement
                if SAM_REFINE:
                    raster_cache = OUTPUT_DIR / ".agribound_cache"
                    raster_candidates = sorted(raster_cache.glob(f"*{source}*{year}*.tif"))
                    if raster_candidates:
                        print(f"    SAM2 refining {len(source_gdf)} fields...")
                        sam_config = AgriboundConfig(
                            source=source,
                            engine="dinov3",
                            year=year,
                            study_area=study_area_path,
                            output_path=str(output_path),
                            engine_params={
                                "sam_model": SAM_MODEL,
                                "sam_batch_size": SAM_BATCH_SIZE,
                            },
                            device="auto",
                        )
                        source_gdf = refine_boundaries(
                            source_gdf,
                            str(raster_candidates[0]),
                            sam_config,
                        )
                        source_gdf = smooth_polygons(source_gdf, iterations=3)
                        source_gdf = simplify_polygons(source_gdf, tolerance=2.0)

                # Write final only after all steps complete
                source_gdf.to_file(output_path, driver="GPKG", layer="fields")
                if pipeline_path.exists():
                    pipeline_path.unlink()

            all_results[year][source] = source_gdf
            print(f"  {tag}: {len(source_gdf)} fields")
        except Exception as exc:
            print(f"  {tag}: FAILED — {exc}")

## Phase 2: Grand Ensemble

In [ ]:
from agribound.engines.ensemble import EnsembleEngine
from agribound.postprocess import filter_polygons

ensemble_results = {}

for year in YEARS:
    year_results = all_results.get(year, {})
    if len(year_results) < 2:
        print(f"{year}: only {len(year_results)} source(s), skipping ensemble.")
        if year_results:
            ensemble_results[year] = next(iter(year_results.values()))
        continue

    output_path = OUTPUT_DIR / f"fields_ensemble_dinov3_{year}.gpkg"
    if output_path.exists():
        ensemble_results[year] = gpd.read_file(output_path)
        print(f"{year}: loaded cached ensemble.")
        continue

    print(f"{year}: merging {len(year_results)} sources...", end=" ")
    ensemble_gdf = EnsembleEngine._merge_vote(year_results, VOTE_THRESHOLD)
    ensemble_gdf = filter_polygons(ensemble_gdf, min_area_m2=2500)
    ensemble_gdf = smooth_polygons(ensemble_gdf, iterations=3)
    ensemble_gdf = simplify_polygons(ensemble_gdf, tolerance=2.0)

    if ensemble_gdf.crs is not None and str(ensemble_gdf.crs) != OUTPUT_CRS:
        ensemble_gdf = ensemble_gdf.to_crs(OUTPUT_CRS)
    ensemble_gdf.to_file(output_path, driver="GPKG", layer="fields")
    ensemble_results[year] = ensemble_gdf
    print(f"{len(ensemble_gdf)} fields")

## Phase 3: Evaluation

In [ ]:
header = f"{'Year':<6} {'Source':<20} {'Fields':>6} {'F1':>6} {'IoU':>6} {'P':>6} {'R':>6}"
print(header)
print(f"{'-' * 6} {'-' * 20} {'-' * 6} {'-' * 6} {'-' * 6} {'-' * 6} {'-' * 6}")

for year in YEARS:
    for source, source_gdf in sorted(all_results.get(year, {}).items()):
        try:
            m = evaluate(source_gdf, county_gdf)
            print(
                f"{year:<6} {source:<20} {len(source_gdf):>6} "
                f"{m['f1']:.3f} {m['iou_mean']:.3f} "
                f"{m['precision']:.3f} {m['recall']:.3f}"
            )
        except Exception:
            pass

    ens = ensemble_results.get(year)
    if ens is not None:
        try:
            m = evaluate(ens, county_gdf)
            print(
                f"{year:<6} {'** ENSEMBLE **':<20} {len(ens):>6} "
                f"{m['f1']:.3f} {m['iou_mean']:.3f} "
                f"{m['precision']:.3f} {m['recall']:.3f}"
            )
        except Exception:
            pass

## Phase 4: Visualization

In [ ]:
from agribound.visualize import show_comparison

if ensemble_results:
    latest_year = max(ensemble_results.keys())
    m = show_comparison(
        [ensemble_results[latest_year], county_gdf],
        labels=[f"DINOv3+SAM2 Ensemble ({latest_year})", "NMOSE Reference"],
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_ensemble_vs_reference.html"),
    )
    m